<a href="https://colab.research.google.com/github/lalithyadev/mini_project/blob/main/STUDENT_PERFORMANCE_PREDICT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import pandas as pd

df = pd.read_csv("/content/student_performance (1).csv")

df.shape
df

,Study_Hours_Per_Week,Attendance_Percentage,Previous_Score,Sleep_Hours_Per_Day,Practice_Papers_Solved,Extracurricular_Activities,Internet_Access,Parent_Education_Level,Class_Participation,Performance_Score
0,1,68.8,82,6.2,4,Yes,Yes,Postgraduate,4,87.0
1,7,75.5,42,7.8,2,No,Yes,School,3,89.9
2,6,85.3,50,8.6,8,No,Yes,Undergraduate,2,94.0
3,4,74.5,84,8.4,7,No,Yes,Postgraduate,2,100.0
4,4,78.7,88,7.4,1,Yes,Yes,Undergraduate,4,84.5
...,...,...,...,...,...,...,...,...,...,...
9995,4,86.0,48,7.0,6,No,Yes,School,1,59.5
9996,8,74.5,39,8.6,8,No,No,Undergraduate,2,91.7
9997,3,72.9,70,8.0,6,Yes,Yes,School,1,79.6
9998,5,76.2,93,8.1,0,Yes,Yes,Postgraduate,2,98.3


In [23]:
from google.colab import drive

# Mount Google Drive to /content/drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np
import os
import re
import shutil

# Project folder
PROJECT_DIR = "/content/Student_Performance_ETL"

# Create folders
folders = [
    "raw_data",
    "reports",
    "screenshots",
    "output"
]

for folder in folders:
    os.makedirs(os.path.join(PROJECT_DIR, folder), exist_ok=True)

print("Project folders created successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project folders created successfully!


In [24]:
def extract():

    file_path = "/content/student_performance (1).csv"

    df = pd.read_csv(file_path)

    print("Data Extraction Successful!")
    print("--------------------------------")
    print("Total Records:", df.shape[0])
    print("Total Columns:", df.shape[1])

    # Save raw data
    raw_path = os.path.join(
        PROJECT_DIR,
        "raw_data",
        "student_performance_raw.csv"
    )

    df.to_csv(raw_path, index=False)

    return df


# Execute extraction
raw_df = extract()

raw_df.head()

Data Extraction Successful!
--------------------------------
Total Records: 10000
Total Columns: 10


,Study_Hours_Per_Week,Attendance_Percentage,Previous_Score,Sleep_Hours_Per_Day,Practice_Papers_Solved,Extracurricular_Activities,Internet_Access,Parent_Education_Level,Class_Participation,Performance_Score
0,1,68.8,82,6.2,4,Yes,Yes,Postgraduate,4,87.0
1,7,75.5,42,7.8,2,No,Yes,School,3,89.9
2,6,85.3,50,8.6,8,No,Yes,Undergraduate,2,94.0
3,4,74.5,84,8.4,7,No,Yes,Postgraduate,2,100.0
4,4,78.7,88,7.4,1,Yes,Yes,Undergraduate,4,84.5


In [25]:
def transform(df):

    df = df.copy()

    print("Starting Data Transformation...")
    print("--------------------------------")

    # 1. Standardize column names
    df.columns = [
        re.sub(r"_+", "_",
               re.sub(r"[^a-zA-Z0-9]+", "_", col.strip().lower())
        ).strip("_")
        for col in df.columns
    ]

    # 2. Remove completely empty columns
    df = df.dropna(axis=1, how="all")

    # 3. Remove duplicate records
    df = df.drop_duplicates()

    # 4. Clean text columns
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].astype("string").str.strip()

    # 5. Standardize email addresses
    for col in df.columns:
        if "email" in col:
            df[col] = df[col].str.lower()

    # 6. Convert numeric-like columns into numeric data types
    for col in df.columns:
        if df[col].dtype == "object":
            converted = pd.to_numeric(df[col], errors="coerce")

            if converted.notna().mean() >= 0.8:
                df[col] = converted

    # 7. Handle missing values
    for col in df.columns:

        if pd.api.types.is_numeric_dtype(df[col]):
            if df[col].isna().any():
                df[col] = df[col].fillna(df[col].median())

        else:
            if df[col].isna().any():
                mode_value = df[col].mode()

                if not mode_value.empty:
                    df[col] = df[col].fillna(mode_value.iloc[0])
                else:
                    df[col] = df[col].fillna("Unknown")

    # 8. Validate score columns
    score_columns = [
        col for col in df.columns
        if any(word in col for word in ["score", "marks"])
        and pd.api.types.is_numeric_dtype(df[col])
    ]

    for col in score_columns:
        df = df[(df[col] >= 0) & (df[col] <= 100)]

    # 9. Calculate average score
    if len(score_columns) >= 2:
        df["average_score"] = df[score_columns].mean(axis=1)

    # 10. Create performance category
    if "average_score" in df.columns:

        df["performance_category"] = pd.cut(
            df["average_score"],
            bins=[-np.inf, 40, 60, 80, np.inf],
            labels=[
                "Needs Improvement",
                "Average",
                "Good",
                "Excellent"
            ]
        )

    # 11. Sort students by average score
    if "average_score" in df.columns:
        df = df.sort_values(
            by="average_score",
            ascending=False
        )

    df = df.reset_index(drop=True)

    print("Data Transformation Completed!")
    print("Records after cleaning:", df.shape[0])
    print("Columns after transformation:", df.shape[1])

    return df


# Execute transformation
cleaned_df = transform(raw_df)

cleaned_df.head()

Starting Data Transformation...
--------------------------------
Data Transformation Completed!
Records after cleaning: 10000
Columns after transformation: 12


,study_hours_per_week,attendance_percentage,previous_score,sleep_hours_per_day,practice_papers_solved,extracurricular_activities,internet_access,parent_education_level,class_participation,performance_score,average_score,performance_category
0,8,79.1,100,6.9,8,No,No,Postgraduate,2,100.0,100.0,Excellent
1,5,90.5,100,7.7,3,No,Yes,Undergraduate,1,100.0,100.0,Excellent
2,4,91.4,100,7.7,7,Yes,No,School,4,100.0,100.0,Excellent
3,7,64.8,100,5.7,5,No,Yes,Undergraduate,1,100.0,100.0,Excellent
4,7,70.7,100,6.9,8,No,Yes,Undergraduate,5,100.0,100.0,Excellent


In [26]:
def load(df):

    reports_path = os.path.join(PROJECT_DIR, "reports")
    output_path = os.path.join(PROJECT_DIR, "output")

    os.makedirs(reports_path, exist_ok=True)
    os.makedirs(output_path, exist_ok=True)

    # Report 1: Complete cleaned dataset
    df.to_csv(
        os.path.join(reports_path, "cleaned_data.csv"),
        index=False
    )

    # Save final dataset in output
    df.to_csv(
        os.path.join(output_path, "student_performance_processed.csv"),
        index=False
    )

    # Identify a ranking column
    ranking_col = (
        "average_score"
        if "average_score" in df.columns
        else next(
            (c for c in df.select_dtypes(include=np.number).columns
             if "score" in c or "marks" in c),
            None
        )
    )

    if ranking_col:

        ranked_df = df.sort_values(
            by=ranking_col,
            ascending=False
        )

        # Report 2: Top 10 students
        ranked_df.head(10).to_csv(
            os.path.join(reports_path, "top10_students.csv"),
            index=False
        )

        # Report 3: Bottom 10 students
        ranked_df.tail(10).to_csv(
            os.path.join(reports_path, "bottom10_students.csv"),
            index=False
        )

    # Report 4: Summary statistics
    summary = df.describe(include="all").transpose()

    summary.to_csv(
        os.path.join(reports_path, "summary_report.csv")
    )

    print("All reports generated successfully!")
    print("--------------------------------")
    print("Cleaned dataset saved")
    print("Top 10 students report saved")
    print("Bottom 10 students report saved")
    print("Summary report saved")

    return df


# Execute load
load(cleaned_df)

All reports generated successfully!
--------------------------------
Cleaned dataset saved
Top 10 students report saved
Bottom 10 students report saved
Summary report saved


,study_hours_per_week,attendance_percentage,previous_score,sleep_hours_per_day,practice_papers_solved,extracurricular_activities,internet_access,parent_education_level,class_participation,performance_score,average_score,performance_category
0,8,79.1,100,6.9,8,No,No,Postgraduate,2,100.0,100.00,Excellent
1,5,90.5,100,7.7,3,No,Yes,Undergraduate,1,100.0,100.00,Excellent
2,4,91.4,100,7.7,7,Yes,No,School,4,100.0,100.00,Excellent
3,7,64.8,100,5.7,5,No,Yes,Undergraduate,1,100.0,100.00,Excellent
4,7,70.7,100,6.9,8,No,Yes,Undergraduate,5,100.0,100.00,Excellent
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,1,100.0,37,7.3,0,No,No,School,3,53.4,45.20,Average
9996,1,70.8,38,6.9,0,No,Yes,School,4,51.7,44.85,Average
9997,1,86.0,35,5.3,0,No,No,Postgraduate,3,53.8,44.40,Average
9998,1,74.2,35,5.6,1,Yes,No,Postgraduate,1,52.0,43.50,Average


In [27]:
def run_etl_pipeline():

    print("================================")
    print(" STUDENT PERFORMANCE ETL PIPELINE")
    print("================================")

    # EXTRACT
    raw_data = extract()

    # TRANSFORM
    transformed_data = transform(raw_data)

    # LOAD
    load(transformed_data)

    print("================================")
    print(" ETL PIPELINE EXECUTED SUCCESSFULLY")
    print("================================")

    return transformed_data


# Execute only when the script is run directly
if __name__ == "__main__":
    final_df = run_etl_pipeline()

 STUDENT PERFORMANCE ETL PIPELINE
Data Extraction Successful!
--------------------------------
Total Records: 10000
Total Columns: 10
Starting Data Transformation...
--------------------------------
Data Transformation Completed!
Records after cleaning: 10000
Columns after transformation: 12
All reports generated successfully!
--------------------------------
Cleaned dataset saved
Top 10 students report saved
Bottom 10 students report saved
Summary report saved
 ETL PIPELINE EXECUTED SUCCESSFULLY


In [28]:
readme_content = """
# Student Performance Data ETL Pipeline

## 1. Project Objective

To build an ETL pipeline that extracts raw student
performance data, cleans and transforms it, and
loads analysis-ready data for future Data Science projects.

## 2. Data Source

Source: Student Performance CSV dataset

File: student_performance (1).csv

Format: CSV

Structure: Structured data

The dataset contains student information and academic
performance details.

Update Frequency: Static dataset. Updates require a new
CSV file to be supplied.

## 3. ETL Workflow

Data Source
    |
    v
Extract
    |
    v
Validate Data
    |
    v
Clean Data
    |
    v
Transform Data
    |
    v
Generate Reports
    |
    v
Load Processed Data

## 4. Transformations Performed

1. Standardize column names.
2. Remove empty columns.
3. Remove duplicate records.
4. Clean text values.
5. Standardize email addresses.
6. Convert numeric data types.
7. Handle missing values.
8. Validate score values.
9. Calculate average scores.
10. Create performance categories.
11. Sort students by average score.

## 5. Reports Generated

- Cleaned dataset
- Top 10 students
- Bottom 10 students
- Summary statistics

## 6. Challenges Faced

- Handling missing values.
- Removing duplicate records.
- Converting incorrect data types.
- Maintaining data consistency.
- Creating meaningful performance categories.

## 7. Learning Outcomes

- Understanding ETL architecture.
- Working with Pandas.
- Data cleaning and preprocessing.
- Data validation.
- Generating analytical reports.
- Organizing Python projects.

## 8. Future Improvements

- Integrate a live student data API.
- Create interactive dashboards.
- Add advanced statistical analysis.
- Build machine learning prediction models.
- Schedule automatic ETL execution.
"""

readme_path = os.path.join(PROJECT_DIR, "README.md")

with open(readme_path, "w") as file:
    file.write(readme_content)

print("README.md created successfully!")

README.md created successfully!
